================================================================================
模块 10：量化回测避坑指南与实盘系统工程 · 第 2/4 课
第 2 课：真实微观撮合仿真：滑点、排队模型与交易摩擦成本 (Microstructure Matching & Slippage)
================================================================================

【课程定位与投决会核心心智模型】
::: callout-note
### 📌 投决会核心心智模型：零滑点与无限流动性是回测中最隐蔽的自欺欺人
业余量化回测最致命的玩具级假设有二：“零滑点无限流动性”与“静态无门槛交易费”。现实中，任何策略指令送入交易所订单簿（LOB）都必须经历物理排队与对手盘消耗。大额订单不可避免地产生非线性市场冲击（Market Impact）；而中国 A 股特有的“券商佣金最低 5 元”、“双边十万分之一过户费”、“卖出单向千分之零点五印花税”以及涨跌停一字封板不可成交，共同构成了侵蚀高频高换手策略真实收益的摩擦黑洞。
:::

- **[平方根市场冲击定律（Square-Root Law）]**：根据 Almgren-Chriss 与微观结构理论，市场冲击成本与订单量占 Bar 成交量的平方根成正比。资金规模越大，单位滑点呈非线性加速恶化。
- **[订单排队与容量上限截断]**：在限价订单簿中，单 Bar 撮合量受制于参与率上限（通常 $\alpha = 10\% \sim 15\%$）。超过容量的订单必须部分成交（Partial Fill）并结转至下一 Bar，严禁瞬时全额无感吃单。
- **[极端行情物理封死判定]**：一字涨停板买单封死（成交为 0）、一字跌停板卖单封死（出逃为 0）、停牌拒绝撮合；任何假设在跌停板能够自由市价止损的回测均为造假。

【微观机理与三位一体就近图文剖析】

锚点指引：下图展示了限价订单簿（LOB）微观排队机制与参与率上限截断流程。

```mermaid
flowchart TD
    Order["策略提交大额订单 (Order Volume)"] --> Check{"检查 Bar 状态与流动性"}
    Check -->|停牌 Volume=0 或 is_suspended| Reject["拒单: SUSPENSION (无法成交)"]
    Check -->|一字涨停且买入 / 一字跌停且卖出| LimitLock["拒单: LIMIT_LOCK (封板排队在队尾)"]
    Check -->|正常连续交易| Match["计算容量上限: Available = Bar_Volume × α (10%)"]
    Match --> Fill["本次成交 = min(剩余委托量, Available) ➔ 剩余结转下一期"]
```

- **[要点 1：价格优先与时间优先的残酷性]**：当策略大单到达盘口时，队列前排早已堆积了做市商与快速通道挂单，策略订单排在队尾无法瞬时消化。
- **[要点 2：参与率上限防范自成交易]**：若某股票 1 分钟仅成交 500 手，策略下单 2000 手若全部以当前价成交，本质是凭空虚构了 1500 手对手盘流动性。
- **[要点 3：部分成交状态流转]**：未成交部分保持 `PARTIALLY_FILLED` 状态，随时间序列推移继续等待后续 Bar 的流动性释放。

锚点指引：下图拆解了 A 股全套法定交易税费漏斗与非线性冲击滑点构成。

```mermaid
flowchart LR
    Trade["单笔成交金额 Amount = P_exec × Q_fill"] --> Fee1["券商佣金: max(5.0, Amount × 0.00025) [双边]"]
    Trade --> Fee2["过户费: Amount × 0.00001 [双边]"]
    Trade --> Fee3["印花税: Amount × 0.0005 [仅卖出]"]
    Trade --> Slip["冲击滑点: Amount × [Base_Slippage + η·√(Q_fill / Bar_Vol)]"]
    Fee1 & Fee2 & Fee3 & Slip --> Total["总摩擦损耗 Total Friction = 佣金 + 过户费 + 印花税 + 滑点"]
```

- **[要点 1：最低 5 元佣金的散户刺客]**：高频小单交易中，数千元成交额的实际佣金率可能高达千分之二（5元 / 2500元），远超名义万 2.5。
- **[要点 2：印花税的单边惩罚]**：卖出时万分之五的印花税属于不可避免的财政硬扣税，直接抬高了每一次换仓的获利门槛。
- **[要点 3：平方根滑点的资金容量天花板]**：随着策略资产规模膨胀，滑点开销呈超线性上升，导致策略出现著名的“规模是 Alpha 的敌人”容量衰竭。

【数学与估值模型】
真实 A 股微观撮合成交价与滑点公式（Almgren-Chriss）：
$$
\text{Slippage} = \text{Base\_Slippage} + \eta \cdot \sqrt{\frac{\text{Order\_Volume}}{\text{Bar\_Volume}}}
$$
$$
P_{\text{exec}} = \begin{cases}
P_{\text{bench}} \times (1 + \text{Slippage}), & \text{Direction} = \text{'BUY'} \\
P_{\text{bench}} \times (1 - \text{Slippage}), & \text{Direction} = \text{'SELL'}
\end{cases}
$$
变量定义域与参数约束：
- $\text{Base\_Slippage} \in [0.0001, 0.0005]$：通常设为 $0.0002$（2 bps，覆盖买卖最小跳价差）；
- $\eta \in [0.02, 0.15]$：市场冲击敏感系数，日频大盘取 $0.03$，分钟频小盘取 $0.08$；
- $\text{Fill\_Volume} \le \text{Bar\_Volume} \times \alpha$，$\alpha \in [0.05, 0.15]$。

A 股法定交易税费方程：
$$
\text{Commission} = \max(5.0, \; \text{Trade\_Amount} \times 0.00025)
$$
$$
\text{Transfer\_Fee} = \text{Trade\_Amount} \times 0.00001
$$
$$
\text{Stamp\_Duty} = \begin{cases}
\text{Trade\_Amount} \times 0.0005, & \text{Direction} = \text{'SELL'} \\
0.0, & \text{Direction} = \text{'BUY'}
\end{cases}
$$

::: callout-warning
### ⚠️ 投决会风控红线与认知陷阱防范
1. **高换手率的摩擦复利吞噬**：年化单边换手率 50 倍的策略，一年仅手续费与双边滑点即可吞噬 10%~15% 的总收益，回测中未加税费的高夏普属于虚无泡沫。
2. **限价单逆向选择（Adverse Selection）**：回测中挂单买入若轻易全额成交，实盘中通常意味着有剧烈的大卖单正在向下击穿盘口，成交后大概率立即面临浮亏。
3. **停牌与一字跌停不可撤销**：持有标的一旦暴雷停牌或一字跌停封死，实盘中卖单只能排在数亿封单队尾，回测中任何当日平仓成功的记录一律视为严重合规事故。
:::

【思考题】
Q1: 为什么小市值高换手策略（例如年化单边换手率 50~100 倍）在扣除万分之五滑点与真实交易成本后，纸面上 80% 的年化超额收益往往会断崖式暴跌至负收益？请从摩擦成本复利损耗的角度定量推导。
Q2: 在基于分钟 Bar 的事件驱动回测中，如果某个 Bar 内限价买单价格为 10.20 元，而该 Bar 的价格区间为 [Low=10.00, High=10.50]，真实市场成交量仅为 500 手（50,000 股），策略下单量为 2,000 手（200,000 股）。为什么绝不能允许这 2,000 手在当前 Bar 全部以 10.20 元成交？如何设计合理的局部成交与排队结转机制？
Q3: 什么是撮合中的“逆向选择（Adverse Selection）”？为什么在回测中限价单（Limit Order）成交往往意味着当前行情正朝着对该订单不利的方向剧烈运动？如何在仿真中刻画这一微观结构现象？

【考核与通过规则】
- 代码填空并通过所有自动化单元测试断言：4 分
- Q1~Q3 问答题深入推导并符合投决会专业标准：各 2 分（共 6 分）
- 综合通过线：8/10 分


In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple
import math


@dataclass
class Order:
    """订单对象"""
    order_id: str
    symbol: str
    direction: str       # 'BUY' 或 'SELL'
    order_type: str      # 'MARKET' (市价单) 或 'LIMIT' (限价单)
    price: float         # 委托限价 (市价单可传 0.0)
    volume: int          # 委托总股数 (100股的整数倍)
    filled_volume: int = 0
    status: str = 'PENDING'  # 'PENDING', 'PARTIALLY_FILLED', 'FILLED', 'REJECTED'


@dataclass
class BarData:
    """行情 Bar 对象"""
    symbol: str
    open: float
    high: float
    low: float
    close: float
    volume: int          # 市场该 Bar 真实成交股数
    limit_up: float      # 当日涨停价
    limit_down: float    # 当日跌停价
    is_suspended: bool = False


@dataclass
class ExecutionReport:
    """撮合成交回报"""
    order_id: str
    status: str
    filled_this_bar: int     # 本次 Bar 撮合成交股数
    cumulative_filled: int   # 累计已成交总股数
    remaining_volume: int    # 剩余未成交股数
    executed_price: float    # 本次成交均价 (包含滑点)
    commission: float        # 券商佣金 (含最低5元保底)
    transfer_fee: float      # 过户费
    stamp_duty: float        # 印花税 (仅卖出)
    slippage_cost: float     # 相比基准价产生的绝对滑点损失金额
    total_friction: float    # 总摩擦成本 = 佣金 + 过户费 + 印花税 + 滑点损失


class EventDrivenMatchingEngine:
    """
    工业级事件驱动微观撮合引擎模拟器:
    支持真实 A 股费用结构、流动性容量上限截断、平方根市场冲击滑点与极端行情不可成交判定。
    """

    def __init__(
        self,
        commission_rate: float = 0.00025,
        min_commission: float = 5.0,
        transfer_fee_rate: float = 0.00001,
        stamp_duty_rate: float = 0.0005,
        base_slippage: float = 0.0002,
        impact_coef: float = 0.05,
        max_volume_ratio: float = 0.10
    ):
        self.commission_rate = commission_rate
        self.min_commission = min_commission
        self.transfer_fee_rate = transfer_fee_rate
        self.stamp_duty_rate = stamp_duty_rate
        self.base_slippage = base_slippage
        self.impact_coef = impact_coef
        self.max_volume_ratio = max_volume_ratio

    def calculate_transaction_fees(self, trade_amount: float, direction: str) -> Tuple[float, float, float]:
        """
        计算法定交易规费：
        1. 券商佣金 (双边，最低 5 元)
        2. 过户费 (双边)
        3. 印花税 (仅卖出)
        """
        # TODO: 计算法定规费
        pass

    def calculate_impact_slippage(
        self,
        benchmark_price: float,
        direction: str,
        order_volume: int,
        bar_volume: int
    ) -> Tuple[float, float]:
        """
        计算平方根市场冲击滑点。
        """
        # TODO: 计算平方根冲击滑点
        pass

    def match(self, order: Order, bar: BarData) -> ExecutionReport:
        """
        执行单 Bar 粒度事件撮合逻辑。
        """
        # TODO: 执行撮合、排队截断与报表生成
        pass


if __name__ == "__main__":
    print("Lesson 02 (Slippage Matching Engine) Stub Initialized.")
